In [1]:
# Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score,
    accuracy_score,
    brier_score_loss)

import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from tqdm import tqdm   

In [7]:
# Define temporal train/test split (w/o leakage)

def temporal_train_test_split(
    df: pd.DataFrame,
    time_col: int,
    class_col: str,
    train_frac: float = 0.8):
    
    # Sort df by time_col, then do an 80/20 temporal split.
    # Returns: X_train, y_train, X_test, y_test
    
    # Sort by time ascending
    df_sorted = df.sort_values(by=time_col).reset_index(drop=True)

    # Compute split index
    split_idx = int(len(df_sorted) * train_frac)

    # Split
    train_df = df_sorted.iloc[:split_idx].copy()
    test_df  = df_sorted.iloc[split_idx:].copy()

    # Separate features and target
    X_train = train_df.drop(columns=[class_col])
    y_train = train_df[class_col].copy()

    X_test  = test_df.drop(columns=[class_col])
    y_test  = test_df[class_col].copy()

    return X_train, y_train, X_test, y_test, df_sorted


In [8]:
# Load dataset 

df = pd.read_csv("D:/Final Year/App Domains/Project/datasets/Synthetic_Financial_Dataset.csv")

# Set time col and then the target col 
time_col   = "Time"  
target_col = "Class" 

# Quick validation
print(df[[time_col, target_col]].head())

KeyError: "None of [Index(['Time', 'Class'], dtype='object')] are in the [columns]"

In [ ]:
# Temporal split and model training for dataset 

# Drop rows with missing time or target if necessary
df_clean = df.dropna(subset=[time_col, target_col]).copy()

# Temporal split
X_train, y_train, X_test, y_test, df_sorted = temporal_train_test_split(
    df_clean,
    time_col,
    target_col,
    train_frac=0.8
)

# Select feature columns (exclude time and target)
feature_cols = [c for c in X_train.columns if c != [time_col, target_col]]

X_train_feat = X_train[feature_cols]
X_test_feat  = X_test[feature_cols]

# Define pipeline (scaling + logistic regression as example) ******* CHANGES 
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

# Fit model
pipeline.fit(X_train_feat, y_train)

# Quick baseline performance on test set
y_test_pre_p = pipeline.predict_proba(X_test_feat)[:, 1]
y_test_pre = (y_test_pre_p >= 0.5).astype(int)

auc  = roc_auc_score(y_test, y_test_pre_p)
acc  = accuracy_score(y_test, y_test_pre)
brier = brier_score_loss(y_test, y_test_pre_p)

print(f"Dataset 1 - Baseline temporal test AUC:   {auc:.3f}")
print(f"Dataset 1 - Baseline temporal test Acc:   {acc:.3f}")
print(f"Dataset 1 - Baseline temporal test Brier: {brier:.3f}")


In [ ]:
# Function to simulate real-time predictions on a dataset


def simulate_realtime_predictions(
    df_sorted: pd.DataFrame,
    time_col: str,
    target_col: str,
    feature_cols: list,
    model,
    window_size: int = 50
):
    """
    Simulate real-time predictions row-by-row with a progress bar.
    """

    df_sim = df_sorted.dropna(subset=[time_col, target_col]).copy()

    y_true_all = df_sim[target_col].values
    X_all      = df_sim[feature_cols]
    time_vals  = df_sim[time_col].values

    pred_probs = []
    pred_labels = []
    metrics_records = []

    
    for i in tqdm(range(len(df_sim)), desc="Simulating predictions"):
        x_row = X_all.iloc[i:i+1]
        proba = model.predict_proba(x_row)[:, 1][0]
        label_pred = int(proba >= 0.5)

        pred_probs.append(proba)
        pred_labels.append(label_pred)

        if i + 1 >= window_size:
            y_window = y_true_all[i+1-window_size:i+1]
            p_window = np.array(pred_probs[i+1-window_size:i+1])
            l_window = np.array(pred_labels[i+1-window_size:i+1])

            from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss

            try:
                auc = roc_auc_score(y_window, p_window)
            except ValueError:
                auc = np.nan

            acc   = accuracy_score(y_window, l_window)
            brier = brier_score_loss(y_window, p_window)

            metrics_records.append({
                "end_index": i,
                "end_time_value": time_vals[i],
                "auc": auc,
                "accuracy": acc,
                "brier": brier
            })

    log_df = pd.DataFrame({
        "time_value": time_vals,
        "y_true": y_true_all,
        "y_pred_prob": pred_probs,
        "y_pred": pred_labels
    })

    metrics_df = pd.DataFrame(metrics_records)
    return log_df, metrics_df

In [ ]:
# Apply real-time simulation to dataset 

log_df, metrics_df = simulate_realtime_predictions(
    df_sorted,
    time_col,
    target_col,
    feature_cols,
    model=pipeline,
    window_size=50)

In [ ]:
log_df.head()
metrics_df.head()

In [ ]:
# Plot rolling AUC, accuracy, Brier over time for dataset 

plt.figure(figsize=(10, 5))
plt.plot(metrics_df["end_time_value"], metrics_df["auc"], label="AUC")
plt.xlabel("Time")
plt.ylabel("AUC")
plt.title("CC Dataset: Rolling AUC over time")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(metrics_df["end_time_value"], metrics_df_1["accuracy"], label="Accuracy")
plt.xlabel("Time")
plt.ylabel("Accuracy")
plt.title("CC Dataset: Rolling Accuracy over time")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(metrics_df_["end_time_value"], metrics_df_1["brier"], label="Brier score")
plt.xlabel("Time")
plt.ylabel("Brier score")
plt.title("CC Dataset: Rolling Brier score over time")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## XG Boost Analysis 

Alwadain et. al 2023


In [ ]:
pipeline_xgb = Pipeline([
        ("xgb", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",      # needed to silence warnings
        n_jobs=-1,
        scale_pos_weight=10         # <-- adjust for your class imbalance
    ))
])


In [ ]:
pipeline_xgb.fit(X_train_feat, y_train)


In [ ]:
y_test_pred_proba_xgb = pipeline_xgb.predict_proba(X_test_feat)[:, 1]
y_test_pred_xgb       = (y_test_pred_proba_xgb >= 0.5).astype(int)

auc_xgb   = roc_auc_score(y_test, y_test_pred_proba_xgb)
acc_xgb   = accuracy_score(y_test, y_test_pred_xgb)
brier_xgb = brier_score_loss(y_test, y_test_pred_proba_xgb)

print(f"XGBoost - temporal test AUC:   {auc_xgb:.2f}")
print(f"XGBoost - temporal test Acc:   {acc_xgb:.2f}")
print(f"XGBoost - temporal test Brier: {brier_xgb:.2f}")


In [ ]:
print("LogReg vs XGBoost (test set)")
print(f"AUC   : {auc_1:.3f}  ->  {auc_xgb:.3f}")
print(f"Acc   : {acc_1:.3f}  ->  {acc_xgb:.3f}")
print(f"Brier : {brier_1:.3f}  ->  {brier_xgb:.3f}")


In [ ]:
log_df_xgb, metrics_df_xgb = simulate_realtime_predictions(
    df_sorted,
    time_col,
    target_col,
    feature_cols,
    model=pipeline_xgb,
    window_size=50)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(metrics_df["end_time_value"], metrics_df["auc"], label="LogReg AUC")
plt.plot(metrics_df_xgb["end_time_value"], metrics_df_xgb["auc"], label="XGBoost AUC", alpha=0.7)
plt.xlabel("Time")
plt.ylabel("AUC")
plt.title("Rolling AUC over time: LogReg vs XGBoost")
plt.legend()
plt.tight_layout()
plt.show()
